# Попередня обробка даних (Spotify)

**Інструкція:** Покладіть сирий файл `spotify_dataset.csv` до каталогу `../data/raw/` відносно цього ноутбука.
Всі проміжні та фінальні файли пишуться у `../data/processed/` та `reports/`.


In [3]:
import shutil
from pathlib import Path
from typing import Dict, List

import pandas as pd
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)        


In [4]:
NUMERIC_SPARK_TYPES = {"double", "float", "int", "bigint", "long", "decimal", "short"}
PROTECTED_COLUMNS = {
    "song_id",
    "song",
    "Artist(s)",
    "Genre",
    "Album",
    "Release Date",
    "Length_seconds",
    "emotion",
    "Popularity",
    "Tempo",
    "Energy",
    "Danceability",
    "Positiveness",
    "Speechiness",
    "Liveness",
    "Acousticness",
    "Instrumentalness",
    "Key",
    "Loudness (db)",
    "Good for Party",
    "Good for Work/Study",
    "Good for Relaxation/Meditation",
    "Good for Exercise",
    "Good for Running",
    "Good for Yoga/Stretching",
    "Good for Driving",
}

def write_single_csv(df: DataFrame, output_path: Path) -> None:
    temp_dir = output_path.parent / f"{output_path.stem}_tmp"
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(str(temp_dir))
    part_files = list(temp_dir.glob("part-*.csv"))
    if not part_files:
        raise FileNotFoundError(f"Не знайдено CSV-шард для {output_path}")
    if output_path.exists():
        output_path.unlink()
    part_files[0].rename(output_path)
    shutil.rmtree(temp_dir)

def profile_columns(df: DataFrame, total_rows: int) -> pd.DataFrame:
    profiles = []
    for name, dtype in df.dtypes:
        column = F.col(name)
        missing_condition = column.isNull()
        if dtype in NUMERIC_SPARK_TYPES:
            missing_condition = missing_condition | F.isnan(column)
        missing = df.select(
            F.sum(F.when(missing_condition, 1).otherwise(0)).alias(name)
        ).collect()[0][name]
        distinct = df.select(name).distinct().count()
        fill_rate = 0.0 if total_rows == 0 else (total_rows - missing) / total_rows
        distinct_ratio = 0.0 if total_rows == 0 else distinct / total_rows
        drop_recommendation = (fill_rate < 0.6) or (distinct_ratio < 0.01)
        profiles.append({
            "column": name,
            "dtype": dtype,
            "missing": int(missing),
            "fill_rate": round(fill_rate, 4),
            "distinct_values": int(distinct),
            "distinct_ratio": round(distinct_ratio, 4),
            "drop_recommendation": bool(drop_recommendation),
        })
    return pd.DataFrame(profiles).sort_values(by="fill_rate", ascending=True)


In [ ]:
DATA_ROOT = Path('..') / 'data'
RAW_DATA_PATH = DATA_ROOT / 'raw' / 'spotify_dataset.csv'
PROCESSED_DIR = DATA_ROOT / 'processed'
REPORTS_DIR = Path('reports')
for path in (DATA_ROOT / 'raw', PROCESSED_DIR, REPORTS_DIR):
    path.mkdir(parents=True, exist_ok=True)
print(f'Оброблені файли будуть у: {PROCESSED_DIR}')
print(f'Звіти авто-генеруються у: {REPORTS_DIR}')


In [ ]:
SPOTIFY_SCHEMA = StructType([
    StructField('Artist(s)', StringType(), True),
    StructField('song', StringType(), True),
    StructField('text', StringType(), True),
    StructField('Length', StringType(), True),
    StructField('emotion', StringType(), True),
    StructField('Genre', StringType(), True),
    StructField('Album', StringType(), True),
    StructField('Release Date', StringType(), True),
    StructField('Key', StringType(), True),
    StructField('Tempo', FloatType(), True),
    StructField('Loudness (db)', StringType(), True),
    StructField('Time signature', StringType(), True),
    StructField('Explicit', StringType(), True),
    StructField('Popularity', IntegerType(), True),
    StructField('Energy', FloatType(), True),
    StructField('Danceability', FloatType(), True),
    StructField('Positiveness', FloatType(), True),
    StructField('Speechiness', FloatType(), True),
    StructField('Liveness', FloatType(), True),
    StructField('Acousticness', FloatType(), True),
    StructField('Instrumentalness', FloatType(), True),
    StructField('Good for Party', IntegerType(), True),
    StructField('Good for Work/Study', IntegerType(), True),
    StructField('Good for Relaxation/Meditation', IntegerType(), True),
    StructField('Good for Exercise', IntegerType(), True),
    StructField('Good for Running', IntegerType(), True),
    StructField('Good for Yoga/Stretching', IntegerType(), True),
    StructField('Good for Driving', IntegerType(), True),
    StructField('Good for Social Gatherings', IntegerType(), True),
    StructField('Good for Morning Routine', IntegerType(), True),
    StructField('Similar Artist 1', StringType(), True),
    StructField('Similar Song 1', StringType(), True),
    StructField('Similarity Score 1', FloatType(), True),
    StructField('Similar Artist 2', StringType(), True),
    StructField('Similar Song 2', StringType(), True),
    StructField('Similarity Score 2', FloatType(), True),
    StructField('Similar Artist 3', StringType(), True),
    StructField('Similar Song 3', StringType(), True),
    StructField('Similarity Score 3', FloatType(), True),
])


In [ ]:
spark = (
    SparkSession.builder
    .appName('SpotifyDataPreprocessing')
    .config('spark.driver.memory', '4g')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark

In [ ]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f'Не знайдено {RAW_DATA_PATH}. Покладіть файл і перезапустіть цю комірку.')
df_raw = (
    spark.read.csv(
        str(RAW_DATA_PATH),
        header=True,
        schema=SPOTIFY_SCHEMA,
        multiLine=True,
        quote='"',
        escape='"',
        mode='PERMISSIVE',
    )
)
df_raw = df_raw.cache()
row_count_raw = df_raw.count()
print(f'Попередньо зчитано {row_count_raw} рядків і {len(df_raw.columns)} колонок')

In [ ]:
numeric_columns = [name for name, dtype in df_raw.dtypes if dtype in NUMERIC_SPARK_TYPES]
overview_df = pd.DataFrame([
    {'metric': 'rows_raw', 'value': row_count_raw},
    {'metric': 'columns_total', 'value': len(df_raw.columns)},
    {'metric': 'numeric_columns', 'value': len(numeric_columns)},
])
overview_path = REPORTS_DIR / 'dataset_overview.csv'
overview_df.to_csv(overview_path, index=False)
overview_df

In [ ]:
if numeric_columns:
    numeric_summary = df_raw.select(*numeric_columns).summary()
    numeric_summary_pd = numeric_summary.toPandas()
    numeric_summary_pd.to_csv(PROCESSED_DIR / 'summary_numeric_features.csv', index=False)
    numeric_summary_pd
else:
    print('У наборі немає числових колонок для статистик.')

In [ ]:
df_typed = df_raw
df_typed = df_typed.withColumn(
    'Release Date',
    F.to_date(
        F.regexp_replace(F.col('Release Date'), r'(\d+)(st|nd|rd|th)', r'\1'),
        'd MMMM yyyy',
    ),
)
df_typed = df_typed.withColumn(
    'Loudness (db)',
    F.regexp_replace(F.col('Loudness (db)'), r'[^0-9\-\.]', '').cast(FloatType()),
)
df_typed = df_typed.withColumn(
    'Explicit',
    F.when(F.col('Explicit').isin('True', 'true', '1', 1), F.lit(1)).otherwise(F.lit(0)).cast(IntegerType()),
)
minutes = F.regexp_extract(F.col('Length'), r'(\d+):(\d+)', 1).cast(IntegerType())
seconds = F.regexp_extract(F.col('Length'), r'(\d+):(\d+)', 2).cast(IntegerType())
df_typed = df_typed.withColumn('Length_seconds', minutes * 60 + seconds)
df_typed = df_typed.drop('text')
df_typed = df_typed.withColumn('song_id', F.monotonically_increasing_id())
df_typed = df_typed.cache()
typed_rows = df_typed.count()
print(f'Після приведення типів доступно {typed_rows} рядків у {len(df_typed.columns)} колонках')


In [ ]:
dedup_cols = ['Artist(s)', 'song', 'Album']
df_deduped = df_typed.dropDuplicates(dedup_cols).cache()
deduped_rows = df_deduped.count()
duplicates_removed = typed_rows - deduped_rows
print(f'Вилучено {duplicates_removed} дублікати(-ів)')


In [ ]:
missing_exprs = []
for name, dtype in df_deduped.dtypes:
    condition = F.col(name).isNull()
    if dtype in NUMERIC_SPARK_TYPES:
        condition = condition | F.isnan(F.col(name))
    missing_exprs.append(F.sum(F.when(condition, 1).otherwise(0)).alias(name))
missing_counts = df_deduped.select(missing_exprs).collect()[0].asDict()
missing_stats = pd.DataFrame([
    {
        'column': col,
        'missing': int(val),
        'missing_share': round((val / deduped_rows) if deduped_rows else 0, 4),
    }
    for col, val in missing_counts.items()
]).sort_values(by='missing', ascending=False)
missing_stats.to_csv(REPORTS_DIR / 'missing_values.csv', index=False)
missing_stats.head(10)


In [ ]:
string_impute_cols = [
    'Artist(s)', 'song', 'Genre', 'Album', 'emotion', 'Key',
    'Similar Artist 1', 'Similar Song 1', 'Similar Artist 2', 'Similar Song 2',
    'Similar Artist 3', 'Similar Song 3'
]
numeric_impute_cols = [
    'Tempo', 'Loudness (db)', 'Popularity', 'Energy', 'Danceability',
    'Positiveness', 'Speechiness', 'Liveness', 'Acousticness',
    'Instrumentalness', 'Similarity Score 1', 'Similarity Score 2', 'Similarity Score 3',
    'Good for Party', 'Good for Work/Study', 'Good for Relaxation/Meditation',
    'Good for Exercise', 'Good for Running', 'Good for Yoga/Stretching',
    'Good for Driving', 'Good for Social Gatherings', 'Good for Morning Routine',
    'Length_seconds'
]
df_imputed = df_deduped.fillna('Unknown', subset=string_impute_cols)
df_imputed = df_imputed.fillna(0, subset=numeric_impute_cols)
df_imputed = df_imputed.withColumn(
    'Release Date',
    F.coalesce(F.col('Release Date'), F.to_date(F.lit('2000-01-01'))),
)
df_imputed = df_imputed.cache()
imputed_rows = df_imputed.count()
print(f'Фінальний обсяг {imputed_rows} рядків')        


In [ ]:
feature_profile_df = profile_columns(df_imputed, total_rows=imputed_rows)
feature_profile_path = REPORTS_DIR / 'feature_profile.csv'
feature_profile_df.to_csv(feature_profile_path, index=False)
drop_candidates = [
    row['column']
    for _, row in feature_profile_df.iterrows()
    if row['drop_recommendation'] and row['column'] not in PROTECTED_COLUMNS
]
df_curated = df_imputed.drop(*drop_candidates) if drop_candidates else df_imputed
curated_rows = df_curated.count()
print(f'Відкинуто {len(drop_candidates)} неінформативних колонок: {drop_candidates}')
print(f'У фінальному наборі {curated_rows} рядків і {len(df_curated.columns)} колонок')
feature_profile_df.head(10)
        


In [ ]:
songs_df = df_curated.select('song_id', 'Artist(s)', 'song', 'Genre', 'Album')
release_info_df = df_curated.select('song_id', 'Release Date', 'Length_seconds', 'emotion', 'Popularity')
audio_features_df = df_curated.select(
    'song_id', 'Key', 'Tempo', 'Loudness (db)', 'Energy', 'Danceability',
    'Positiveness', 'Speechiness', 'Liveness', 'Acousticness', 'Instrumentalness'
)
activities_df = df_curated.select(
    'song_id', 'Good for Party', 'Good for Work/Study', 'Good for Relaxation/Meditation',
    'Good for Exercise', 'Good for Running', 'Good for Yoga/Stretching', 'Good for Driving',
    'Good for Social Gatherings', 'Good for Morning Routine'
)
similarities_df = df_curated.select(
    'song_id', 'Similar Artist 1', 'Similar Song 1', 'Similarity Score 1',
    'Similar Artist 2', 'Similar Song 2', 'Similarity Score 2',
    'Similar Artist 3', 'Similar Song 3', 'Similarity Score 3'
)
write_single_csv(df_curated, PROCESSED_DIR / 'spotify_curated.csv')
df_curated.write.mode('overwrite').parquet(str(PROCESSED_DIR / 'spotify_curated.parquet'))
write_single_csv(songs_df, PROCESSED_DIR / 'songs.csv')
write_single_csv(release_info_df, PROCESSED_DIR / 'release_info.csv')
write_single_csv(audio_features_df, PROCESSED_DIR / 'audio_features.csv')
write_single_csv(activities_df, PROCESSED_DIR / 'activities.csv')
write_single_csv(similarities_df, PROCESSED_DIR / 'similar_tracks.csv')

In [ ]:
from IPython.display import Markdown
summary_lines = [
    f'Початковий обсяг: {row_count_raw:,} рядків у {len(df_raw.columns)} колонках.',
    f'Після приведення типів і очищення: {curated_rows:,} рядків.',
    f'Видалено дублікатів: {duplicates_removed}.',
    f'Імпутовано пропуски у {len(missing_stats)} колонках (див. reports/missing_values.csv).',
    f'Відкинуто колонок за низькою інформативністю: {len(drop_candidates)}.',
    'Згенеровано CSV/Parquet у каталозі data/processed та профілі у reports/.',
]
summary_md = '\n'.join(['# Підсумок етапу попередньої обробки'] + [f'- {line}' for line in summary_lines])
(REPORTS_DIR / 'preprocessing_summary.md').write_text(summary_md, encoding='utf-8')
Markdown(summary_md)


In [ ]:
spark.stop()